# 🎯 inzva DLSG#10 — CIFAR-100 Classification (Kaggle Version)
## Complete Educational Pipeline: From Scratch CNN → ResNet18 Transfer Learning

**Milestones:**
1. 🧱 Baseline CNN from Scratch
2. 🛡️ Data Augmentation & Regularization
3. ⚙️ Hyperparameter Tuning
4. 🚀 Transfer Learning with ResNet18 (🏆 best accuracy)
5. 🔍 Interpretability with Captum
6. 🐸 OOD Frog Testing

---
Look for **📝 LEARN:** comments — they explain the *why* behind every technique!

> ⚠️ **Make sure you have added the competition dataset!**  
> Click **Add Input** (right panel) → **Competition** → search `inzva-dlsg10-cv-week-1` → **Add**

## 0️⃣ Setup — GPU Check & Dependencies

In [ ]:
import subprocess, sys

# Install captum without breaking numpy
try:
    import captum
    print('✅ Captum already installed')
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'captum', '--no-deps', '-q'])
    print('✅ Captum installed')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import transforms, models
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import pandas as pd
import os, time, glob
from PIL import Image

try:
    from captum.attr import Occlusion
    CAPTUM_AVAILABLE = True
    print('✅ Captum loaded')
except ImportError:
    CAPTUM_AVAILABLE = False
    print('⚠️ Captum not available')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'\n🖥️  Device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    mem = torch.cuda.get_device_properties(0).total_memory
    print(f'   VRAM: {mem / 1e9:.1f} GB')
print(f'📦 torch={torch.__version__}, numpy={np.__version__}')

## 📂 Discover Competition Data

📝 **LEARN:** Kaggle mounts competition data at `/kaggle/input/<competition-name>/`  
This cell auto-detects the path so you don't have to hardcode it.

In [ ]:
# ============================================================================
# Auto-detect competition data path
# ============================================================================
INPUT_DIR = '/kaggle/input'
WORKING_DIR = '/kaggle/working'

# Find the competition directory (look for train.csv)
COMP_DIR = None
for d in os.listdir(INPUT_DIR):
    check = os.path.join(INPUT_DIR, d)
    if os.path.isdir(check) and os.path.exists(os.path.join(check, 'train.csv')):
        COMP_DIR = check
        break

if COMP_DIR is None:
    print('❌ Competition data not found!')
    print('   Click "Add Input" → "Competition" → search "inzva-dlsg10-cv-week-1" → Add')
else:
    TRAIN_CSV   = os.path.join(COMP_DIR, 'train.csv')
    TRAIN_DIR   = os.path.join(COMP_DIR, 'train_images')
    TEST_DIR    = os.path.join(COMP_DIR, 'test_images')
    SAMPLE_SUB  = os.path.join(COMP_DIR, 'sample_submission.csv')
    FROG_PATH   = os.path.join(COMP_DIR, 'frog.png')
    FROG_FLIP   = os.path.join(COMP_DIR, 'frog-flip.png')
    CLASS_MAP   = os.path.join(COMP_DIR, 'class_label_map.csv')
    
    print(f'✅ Found competition data at: {COMP_DIR}')
    print(f'   Train images: {len(os.listdir(TRAIN_DIR)):,}')
    print(f'   Test images:  {len(os.listdir(TEST_DIR)):,}')
    print(f'   Frog images:  {os.path.exists(FROG_PATH) and os.path.exists(FROG_FLIP)}')
    
    # Load class names
    class_df = pd.read_csv(CLASS_MAP)
    CIFAR100_CLASSES = class_df.sort_values('id')['label_name'].tolist()
    print(f'   Classes: {len(CIFAR100_CLASSES)} → {CIFAR100_CLASSES[:5]}...')
    
    # Load train labels
    train_df = pd.read_csv(TRAIN_CSV)
    print(f'\n📊 Train CSV preview:')
    print(train_df.head())

## 📦 Custom Dataset & Data Loading

📝 **LEARN:** Unlike torchvision's built-in CIFAR-100, this competition gives us raw images + CSV labels.  
We build a custom `Dataset` class that reads images from disk and applies transforms.

In [ ]:
# ============================================================================
# 📝 LEARN: Normalization Constants
# CIFAR stats: for custom CNN models
# ImageNet stats: REQUIRED for pre-trained models (ResNet18 etc.)
# ============================================================================
CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)


class CIFAR100Dataset(Dataset):
    """
    📝 LEARN: Custom Dataset for competition images.
    __getitem__ loads one image, applies transforms, returns (image_tensor, label).
    PyTorch DataLoader calls this to build batches.
    """
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, row['filename'])
        image = Image.open(img_path).convert('RGB')
        label = int(row['class'])
        if self.transform:
            image = self.transform(image)
        return image, label


class TestImageDataset(Dataset):
    """Dataset for unlabeled test images (for submission)."""
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.files = sorted([f for f in os.listdir(img_dir) if f.endswith('.png')])
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        fname = self.files[idx]
        img = Image.open(os.path.join(self.img_dir, fname)).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, fname


def get_transforms(augment=False, image_size=32, use_imagenet_stats=False):
    """📝 LEARN: Transforms = preprocessing pipeline for each image."""
    mean = IMAGENET_MEAN if use_imagenet_stats else CIFAR100_MEAN
    std = IMAGENET_STD if use_imagenet_stats else CIFAR100_STD
    
    if augment:
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomCrop(image_size, padding=4),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.RandomRotation(15),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])


def get_dataloaders(train_df, img_dir, batch_size=64, augment=False,
                    image_size=32, use_imagenet_stats=False, val_split=0.1):
    """📝 LEARN: Create train/val split from training data."""
    train_transform = get_transforms(augment=augment, image_size=image_size,
                                     use_imagenet_stats=use_imagenet_stats)
    val_transform = get_transforms(augment=False, image_size=image_size,
                                   use_imagenet_stats=use_imagenet_stats)
    
    # Reproducible split
    np.random.seed(42)
    indices = np.random.permutation(len(train_df))
    split = int(len(train_df) * val_split)
    
    val_df = train_df.iloc[indices[:split]]
    tr_df = train_df.iloc[indices[split:]]
    
    train_ds = CIFAR100Dataset(tr_df, img_dir, train_transform)
    val_ds = CIFAR100Dataset(val_df, img_dir, val_transform)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    
    print(f'📊 Train: {len(tr_df)} | Val: {len(val_df)} | Batch: {batch_size} | '
          f'Size: {image_size}x{image_size} | Aug: {augment} | '
          f'Norm: {"ImageNet" if use_imagenet_stats else "CIFAR"}')
    return train_loader, val_loader

print('✅ Data utilities loaded!')

## 🏋️ Training Utilities

📝 **LEARN: The Training Loop — Heart of Deep Learning**
1. **FORWARD**: Input → Model → Prediction
2. **LOSS**: Compare prediction vs true label
3. **BACKWARD**: Compute gradients (backpropagation)
4. **UPDATE**: Optimizer adjusts weights
5. **REPEAT** for every batch, for many epochs

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(loader, desc='  Train', leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        _, pred = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()
    return total_loss / total, 100.0 * correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for images, labels in tqdm(loader, desc='  Val', leave=False):
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        _, pred = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()
    return total_loss / total, 100.0 * correct / total

def train_model(model, train_loader, val_loader, criterion, optimizer,
                num_epochs=10, scheduler=None, save_name='best_model.pth'):
    save_path = os.path.join(WORKING_DIR, save_name)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    params = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'\n🚀 Training {num_epochs} epochs | Params: {params:,} ({trainable:,} trainable) | Device: {device}')
    print('=' * 80)
    for epoch in range(num_epochs):
        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        if scheduler:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()
        saved = ''
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
            saved = ' ⭐ SAVED'
        lr = optimizer.param_groups[0]['lr']
        gap = train_acc - val_acc
        gap_warn = f' ⚠️ GAP:{gap:.0f}%' if gap > 20 else ''
        print(f'  [{epoch+1}/{num_epochs}] {time.time()-t0:.0f}s | '
              f'Train: {train_loss:.4f}/{train_acc:.1f}% | '
              f'Val: {val_loss:.4f}/{val_acc:.1f}% | LR: {lr:.6f}{saved}{gap_warn}')
    print(f'\n🏆 Best Validation Accuracy: {best_val_acc:.2f}%')
    return history

def plot_history(history, title='Training'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history['train_loss'], label='Train', color='#FF6B6B', lw=2)
    ax1.plot(history['val_loss'], label='Val', color='#4ECDC4', lw=2)
    ax1.set_title(f'{title} — Loss'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(history['train_acc'], label='Train', color='#FF6B6B', lw=2)
    ax2.plot(history['val_acc'], label='Val', color='#4ECDC4', lw=2)
    ax2.set_title(f'{title} — Accuracy (%)'); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

print('✅ Training utilities loaded!')

---
# 🧱 Milestone 1: Baseline CNN from Scratch

📝 **LEARN: What is a CNN?**
- **Conv2d**: Sliding filter detects patterns (edges, textures)
- **MaxPool2d**: Shrinks image, keeps strongest activations
- **ReLU**: `max(0, x)` — adds non-linearity
- **Goal**: Beat random guessing (1% = 1/100 classes)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(True), nn.MaxPool2d(2, 2))
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(True), nn.MaxPool2d(2, 2))
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(True), nn.MaxPool2d(2, 2))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128*4*4, 256),
            nn.ReLU(True), nn.Linear(256, num_classes))
    def forward(self, x):
        return self.classifier(self.block3(self.block2(self.block1(x))))

print('🧱 Milestone 1: Baseline CNN')
train_loader, val_loader = get_dataloaders(train_df, TRAIN_DIR, batch_size=128, augment=False, image_size=32)
model_m1 = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_m1.parameters(), lr=1e-3)
history_m1 = train_model(model_m1, train_loader, val_loader, criterion, optimizer,
                          num_epochs=15, save_name='m1_baseline.pth')
plot_history(history_m1, 'M1: Baseline CNN')
print(f'\n✅ Best val accuracy: {max(history_m1["val_acc"]):.2f}%')

---
# 🛡️ Milestone 2: Data Augmentation & Regularization

📝 **LEARN: The Overfitting Problem**
- 500 images/class → model can MEMORIZE them
- **Augmentation**: Random flips, crops → "expands" dataset
- **Dropout**: Randomly disables neurons → forces redundant learning

In [ ]:
class AugmentedCNN(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(True), nn.MaxPool2d(2, 2), nn.Dropout2d(0.1))
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(True), nn.MaxPool2d(2, 2), nn.Dropout2d(0.2))
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(True), nn.MaxPool2d(2, 2), nn.Dropout2d(0.3))
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256),
            nn.ReLU(True), nn.AdaptiveAvgPool2d((2, 2)))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(256*2*2, 512),
            nn.ReLU(True), nn.Dropout(0.5), nn.Linear(512, num_classes))
    def forward(self, x):
        return self.classifier(self.block4(self.block3(self.block2(self.block1(x)))))

print('🛡️ Milestone 2: Augmentation + Dropout')
train_loader, val_loader = get_dataloaders(train_df, TRAIN_DIR, batch_size=128, augment=True, image_size=32)
model_m2 = AugmentedCNN().to(device)
optimizer = optim.Adam(model_m2.parameters(), lr=1e-3)
history_m2 = train_model(model_m2, train_loader, val_loader, criterion, optimizer,
                          num_epochs=20, save_name='m2_augmented.pth')
plot_history(history_m2, 'M2: Augmentation + Dropout')
print(f'\n📊 M1: {max(history_m1["val_acc"]):.2f}% → M2: {max(history_m2["val_acc"]):.2f}% ({max(history_m2["val_acc"])-max(history_m1["val_acc"]):+.2f}%)')

---
# ⚙️ Milestone 3: Hyperparameter Tuning

📝 **LEARN:**
- **AdamW**: Adam with properly decoupled weight decay
- **ReduceLROnPlateau**: Auto-reduce LR when val_loss stalls
- **Weight Decay**: Penalizes large weights → smooths decision boundaries

In [ ]:
print('⚙️ Milestone 3: AdamW + Scheduler + Weight Decay')
train_loader, val_loader = get_dataloaders(train_df, TRAIN_DIR, batch_size=128, augment=True, image_size=32)
model_m3 = AugmentedCNN().to(device)
optimizer = optim.AdamW(model_m3.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
history_m3 = train_model(model_m3, train_loader, val_loader, criterion, optimizer,
                          num_epochs=25, scheduler=scheduler, save_name='m3_tuned.pth')
plot_history(history_m3, 'M3: AdamW + Scheduler')
print(f'\n📊 M1: {max(history_m1["val_acc"]):.2f}% | M2: {max(history_m2["val_acc"]):.2f}% | M3: {max(history_m3["val_acc"]):.2f}%')

---
# 🚀 Milestone 4: Transfer Learning with ResNet18

📝 **LEARN: Standing on Giants' Shoulders**
- ResNet18 trained on **1.2M ImageNet images** — already knows edges, textures, shapes
- We replace the final layer for our 100 classes
- **Skip Connections**: `output = F(x) + x` prevents degradation
- ⚠️ **MUST use ImageNet normalization + 224×224 size!**

In [ ]:
print('🚀 Milestone 4: Transfer Learning with ResNet18')

# ⚠️ CRITICAL: ImageNet normalization + 224x224
train_loader, val_loader = get_dataloaders(
    train_df, TRAIN_DIR, batch_size=64, augment=True,
    image_size=224, use_imagenet_stats=True)

model_m4 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
num_features = model_m4.fc.in_features
model_m4.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(num_features, 100))
model_m4 = model_m4.to(device)

# 📝 Differential Learning Rates: low for backbone, high for new head
backbone = [p for n, p in model_m4.named_parameters() if 'fc' not in n]
head = [p for n, p in model_m4.named_parameters() if 'fc' in n]
optimizer = optim.AdamW([
    {'params': backbone, 'lr': 1e-4},
    {'params': head, 'lr': 1e-3}
], weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

history_m4 = train_model(model_m4, train_loader, val_loader, criterion, optimizer,
                          num_epochs=20, scheduler=scheduler, save_name='m4_resnet18.pth')
plot_history(history_m4, 'M4: ResNet18')

print(f'\n📊 All milestones:')
for name, h in [('M1 baseline', history_m1), ('M2 augmented', history_m2),
                ('M3 tuned', history_m3), ('M4 ResNet18', history_m4)]:
    print(f'   {name:15s}: {max(h["val_acc"]):.2f}%')

---
# 🔍 Milestone 5: Interpretability with Captum

📝 **LEARN:** Occlusion slides a grey patch across the image.  
If covering a region DROPS confidence → that region is IMPORTANT for the prediction.

In [ ]:
def denormalize(tensor, mean=IMAGENET_MEAN, std=IMAGENET_STD):
    tensor = tensor.clone()
    for t, m, s in zip(tensor, mean, std):
        t.mul_(s).add_(m)
    return tensor.clamp_(0, 1)

if not CAPTUM_AVAILABLE:
    print('⚠️ Captum not available — skipping interpretability')
else:
    # Load best model
    model_m4.load_state_dict(torch.load(os.path.join(WORKING_DIR, 'm4_resnet18.pth'), map_location=device))
    model_m4.eval()
    
    # Pick 4 random training images for visualization
    test_transform = get_transforms(augment=False, image_size=224, use_imagenet_stats=True)
    np.random.seed(42)
    sample_df = train_df.sample(4, random_state=42)
    sample_ds = CIFAR100Dataset(sample_df, TRAIN_DIR, test_transform)
    
    occlusion = Occlusion(model_m4)
    fig, axes = plt.subplots(4, 3, figsize=(15, 20))
    
    for i in range(4):
        img, true_label = sample_ds[i]
        inp = img.unsqueeze(0).to(device)
        with torch.no_grad():
            out = model_m4(inp)
            pred = out.argmax(1).item()
            conf = torch.softmax(out, 1)[0, pred].item()
        attr = occlusion.attribute(inp, target=pred,
            sliding_window_shapes=(3, 16, 16), strides=(3, 8, 8), baselines=0)
        attr_map = np.mean(np.abs(attr.squeeze().cpu().numpy()), axis=0)
        orig = denormalize(img).permute(1, 2, 0).numpy()
        axes[i, 0].imshow(orig)
        axes[i, 0].set_title(f'True: {CIFAR100_CLASSES[true_label]}'); axes[i, 0].axis('off')
        axes[i, 1].imshow(attr_map, cmap='hot')
        axes[i, 1].set_title(f'Pred: {CIFAR100_CLASSES[pred]} ({conf:.0%})'); axes[i, 1].axis('off')
        axes[i, 2].imshow(orig); axes[i, 2].imshow(attr_map, cmap='hot', alpha=0.5)
        axes[i, 2].set_title(f'Overlay {"✅" if pred == true_label else "❌"}'); axes[i, 2].axis('off')
    plt.suptitle('Milestone 5: Occlusion Attribution', fontsize=16)
    plt.tight_layout(); plt.show()

---
# 🐸 Milestone 6: OOD Frog Testing

📝 **LEARN: Invariance Check** — Does flipping the frog change the prediction?  
If yes → model relies on spatial positioning, not semantic features.

In [ ]:
model_m4.eval()
test_transform = get_transforms(augment=False, image_size=224, use_imagenet_stats=True)

frog_files = [(FROG_PATH, 'frog.png'), (FROG_FLIP, 'frog-flip.png')]
valid_frogs = [(p, n) for p, n in frog_files if os.path.exists(p)]

if not valid_frogs:
    print('⚠️ Frog images not found in competition data')
else:
    fig, axes = plt.subplots(len(valid_frogs), 3, figsize=(15, 5 * len(valid_frogs)))
    if len(valid_frogs) == 1: axes = axes.reshape(1, -1)
    
    predictions = []
    for i, (fpath, fname) in enumerate(valid_frogs):
        img = Image.open(fpath).convert('RGB')
        tensor = test_transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model_m4(tensor)
            probs = torch.softmax(out, 1)[0]
            top5_p, top5_i = probs.topk(5)
        pred = top5_i[0].item()
        predictions.append((fname, pred, top5_p[0].item()))
        print(f'\n📷 {fname}:')
        for j in range(5):
            print(f'   {"👑" if j==0 else "  "} {CIFAR100_CLASSES[top5_i[j]]:20s} {top5_p[j]:.1%}')
        axes[i, 0].imshow(img); axes[i, 0].set_title(fname); axes[i, 0].axis('off')
        axes[i, 1].barh([CIFAR100_CLASSES[idx] for idx in top5_i.cpu()][::-1],
                         top5_p.cpu().numpy()[::-1])
        axes[i, 1].set_title('Top-5'); axes[i, 1].set_xlim(0, 1)
        if CAPTUM_AVAILABLE:
            occ = Occlusion(model_m4)
            attr = occ.attribute(tensor, target=pred,
                sliding_window_shapes=(3, 16, 16), strides=(3, 8, 8), baselines=0)
            attr_map = np.mean(np.abs(attr.squeeze().cpu().numpy()), axis=0)
            disp = denormalize(tensor.squeeze().cpu()).permute(1, 2, 0).numpy()
            axes[i, 2].imshow(disp); axes[i, 2].imshow(attr_map, cmap='hot', alpha=0.5)
            axes[i, 2].set_title(f'Captum: {CIFAR100_CLASSES[pred]}'); axes[i, 2].axis('off')
        else:
            axes[i, 2].text(0.5, 0.5, 'Captum N/A', ha='center', va='center'); axes[i, 2].axis('off')
    plt.tight_layout(); plt.show()
    if len(predictions) >= 2:
        same = predictions[0][1] == predictions[1][1]
        print(f'\n🔄 Invariance: {"✅ Same prediction" if same else "⚠️ Different predictions — spatial bias!"}')

---
# 📤 Generate Kaggle Submission

This creates `submission.csv` in `/kaggle/working/` — Kaggle auto-detects it!

In [ ]:
# Load best model
model_m4.load_state_dict(torch.load(os.path.join(WORKING_DIR, 'm4_resnet18.pth'), map_location=device))
model_m4.eval()

test_transform = get_transforms(augment=False, image_size=224, use_imagenet_stats=True)
test_dataset = TestImageDataset(TEST_DIR, test_transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)
print(f'🧪 Test images: {len(test_dataset)}')

all_names, all_preds = [], []
with torch.no_grad():
    for images, filenames in tqdm(test_loader, desc='Predicting'):
        outputs = model_m4(images.to(device))
        _, preds = torch.max(outputs, 1)
        all_names.extend(filenames)
        all_preds.extend(preds.cpu().numpy())

submission = pd.DataFrame({'filename': all_names, 'class': all_preds})
sub_path = os.path.join(WORKING_DIR, 'submission.csv')
submission.to_csv(sub_path, index=False)

print(f'\n✅ submission.csv saved to {sub_path}')
print(f'   Total predictions: {len(submission)}')
print(f'   Unique classes predicted: {submission["class"].nunique()}')
print(submission.head(10))

---
# 🎉 Done!

| Milestone | Concept | Key Takeaway |
|---|---|---|
| 1 | CNN from scratch | Conv → Pool → Linear pipeline |
| 2 | Augmentation | Fighting overfitting with data diversity |
| 3 | Hyperparameters | LR, optimizer, scheduler, weight decay |
| 4 | Transfer Learning | Pre-trained models are powerful! |
| 5 | Interpretability | Understanding what models "see" |
| 6 | OOD Testing | Real-world robustness |

Click **Submit** in the top right to submit `submission.csv` to the competition! 🚀